# Subjects Independent Experiments

In [1]:
#all subjects separated into 80-20 train-test split, with all samples of each subject in one split exclusively.

In [2]:
## FOR GOOGLE COLLAB ONLY
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
## FOR GOOGLE COLLAB ONLY
#extract EEG & FNIRS data
#cp /content/drive/MyDrive/Data.zip .
#!unzip -qq Data.zip

In this notebook, we will apply EF-Net classifier to Hybrid EEG + fNIRS data with the data split in a Subject Independent setting. 

The current settings in this file will run the EF-Net model. (baseline_model==False). We also have the Deep Learning baselines here, that can be implemented using the 'baseline_model' boolean set to True, and further the 'resnet' boolean which indicates a resnet baseline to be used. If resnet==False, then the VGG baseline shall be implemented.
Deep learning parameters were set according to the table in the appendix of our published paper: [EF-Net: Mental State Recognition by Analyzing Multimodal EEG-fNIRS via CNN](https://doi.org/10.3390/s24061889). For more information on this code, please refer to this paper as well.

The dataset we have used is called: Simultaneous Acquisition of EEG and NIRS during Cognitive Tasks for
an Open Access Dataset,
    and is available here: https://doc.ml.tu-berlin.de/simultaneous_EEG_NIRS/. We are using Dataset C for classification.
    There is 72 fNIRS channels and 30 EEG channels. The dataset has 26 subjects in total. Each subject has data of 60 trials: which includes 3 sessions per subject, each session has 20 trials, each trial has 10 Word Generation tasks, and 10 Baseline tasks. This brings each subject to a total of 60 trials of either WG or BL tasks, performed in random order. Here we will further use those 60 trials to create 360 samples per subject, by using a sliding window technique across each trial.

In [4]:
import numpy as np
import tensorflow as tf
from data_preprocess import extract_subj_indep
import hybrid_model_structures as models
print(tf.config.list_physical_devices('GPU'))

2025-04-30 01:11:51.173541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-30 01:11:51.190837: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-30 01:11:51.196194: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-30 01:11:51.210328: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Inputs - enter here:

In [5]:
#INPUT

#subjects directory path
#The data should be extracted in the same directory as this script.
dir_path_e= "../../Data/EF-WG/Raw/"
dir_path_f = "../../Data/EF-WG/Raw/"

#######################
#Deep Learning settings:

baseline_model=False   #using baselines or EFNet
resnet=False
if baseline_model:
    lr=1e-5
    EPOCHS=100
else:
    lr=5e-5
    EPOCHS=40
BATCH_SIZE = 64

RANDOM_STATE=38

#### Extracting samples from the data

In [6]:
Xeeg_train, Xfnirs_train, Y_train, Xeeg_test, Xfnirs_test, Y_test= extract_subj_indep(dir_path_e,dir_path_f,baseline_model,RANDOM_STATE)

[14, 17, 6, 8, 5, 12, 9, 0, 24, 20, 7, 18, 3, 23, 15, 13, 22, 10, 16, 25, 2, 4, 19, 21, 11, 1]
fnirs len 20245
eeg len 372182
Subject  15 is done. Length of X_train,X_test is now: 360 0 and length of Y_train, Y_test is:  360 0
fnirs len 20295
eeg len 372437
Subject  18 is done. Length of X_train,X_test is now: 720 0 and length of Y_train, Y_test is:  720 0
fnirs len 20145
eeg len 371449
Subject  7 is done. Length of X_train,X_test is now: 1080 0 and length of Y_train, Y_test is:  1080 0
fnirs len 20241
eeg len 371618
Subject  9 is done. Length of X_train,X_test is now: 1440 0 and length of Y_train, Y_test is:  1440 0
fnirs len 20191
eeg len 371874
Subject  6 is done. Length of X_train,X_test is now: 1800 0 and length of Y_train, Y_test is:  1800 0
fnirs len 20210
eeg len 371163
Subject  13 is done. Length of X_train,X_test is now: 2160 0 and length of Y_train, Y_test is:  2160 0
fnirs len 20191
eeg len 371869
Subject  10 is done. Length of X_train,X_test is now: 2520 0 and length of Y_

We now have an array of 360 samples per subject, with a total of 9360 samples as shown across the train and test in output above. Each sample is a 5 second window from the task data, with one second of overlap with the previous sample.  
Secondly, we have an array Y made from the labels (y) array in session_markers from the dataset, and by appending the corresponding classes across the samples taken, we have 360 trial labels per subject in the Y array, totalling 9360 labels across all subjects' trials.

### Standardizing the data

In [7]:
from sklearn.preprocessing import StandardScaler

In [8]:
# Standardize our signals data
# Reshaping the data samples into 2D applying the scaling and then reshaping back
scaler = StandardScaler()
Xeeg_trainS = scaler.fit_transform(Xeeg_train.reshape(-1, Xeeg_train.shape[-1])).reshape(Xeeg_train.shape)
Xeeg_testS = scaler.transform(Xeeg_test.reshape(-1, Xeeg_test.shape[-1])).reshape(Xeeg_test.shape)

Xfnirs_trainS = scaler.fit_transform(Xfnirs_train.reshape(-1, Xfnirs_train.shape[-1])).reshape(Xfnirs_train.shape)
Xfnirs_testS = scaler.transform(Xfnirs_test.reshape(-1, Xfnirs_test.shape[-1])).reshape(Xfnirs_test.shape)

print(Xeeg_trainS.shape)
print(Xfnirs_trainS.shape)

#adjustment to labels to make compatible with binary_crossentropy
Y_train=Y_train[:,0]
Y_test=Y_test[:,0]

(5400, 500, 30)
(5400, 25, 72)


In [9]:
#reshaping data into correct form for input into models.
if baseline_model:
    Xeeg_trainS_3D=Xeeg_trainS
    Xeeg_testS_3D=Xeeg_testS
    Xfnirs_trainS_3D=Xfnirs_trainS
    Xfnirs_testS_3D=Xfnirs_testS
else:
    #reshape into 2D and 1 channel data (3D samples list) for EF-Net.
    Xeeg_trainS_3D=np.reshape(Xeeg_trainS,(Xeeg_trainS.shape[0],Xeeg_trainS.shape[1],Xeeg_trainS.shape[2],1))
    Xeeg_testS_3D=np.reshape(Xeeg_testS,(Xeeg_testS.shape[0],Xeeg_testS.shape[1],Xeeg_testS.shape[2],1))
    Xfnirs_trainS_3D=np.reshape(Xfnirs_trainS,(Xfnirs_trainS.shape[0],Xfnirs_trainS.shape[1],Xfnirs_trainS.shape[2],1))
    Xfnirs_testS_3D=np.reshape(Xfnirs_testS,(Xfnirs_testS.shape[0],Xfnirs_testS.shape[1],Xfnirs_testS.shape[2],1))
    Xeeg_trainS_3D.shape

### Applying a deep neural network to our data

In [10]:
if baseline_model:
    hybridmodel= models.eeg_fnirs_vgg(Xeeg_trainS_3D,Xfnirs_trainS_3D,resnet)
else:
    hybridmodel= models.eeg_fnirs_cnn_v2()

2025-04-30 01:12:12.789971: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 74064 MB memory:  -> device: 0, name: NVIDIA A800 80GB PCIe, pci bus id: 0000:a1:00.0, compute capability: 8.0


In [11]:
hybridmodel.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 30,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 494, 30,   │        256 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 488, 30,   │      7,200 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 482, 30,   │      7,200 │ conv2d_1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 68, 30,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25, 72, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 68, 30,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 22, 72,    │        160 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 68, 30,    │        128 │ dropout[0][0]     │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 19, 72,    │      4,128 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 65, 27,    │     32,832 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 72, 32) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 62, 24,    │     65,600 │ conv2d_3[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 4, 72, 32) │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 59, 21,    │     65,600 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 72, 32) │        128 │ dropout_3[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 14, 5, 64) │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │                   │            │                 

 Total params: 1,757,633 (6.70 MB)

 Trainable params: 1,757,249 (6.70 MB)

 Non-trainable params: 384 (1.50 KB)

In [12]:
from keras.optimizers import Adam

hybridmodel.compile(optimizer=Adam(learning_rate=lr),loss='binary_crossentropy',metrics=['accuracy','Precision','Recall','AUC'])

In [13]:
#save best model weights

checkpoint_filepath = '/tmp/checkpoint.weights.h5'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1)

In [14]:
hybridmodel.fit(x=[Xeeg_trainS_3D, Xfnirs_trainS_3D], y=Y_train, validation_data=([Xeeg_testS_3D, Xfnirs_testS_3D], Y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[model_checkpoint_callback])

Epoch 1/40


/data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_1']. Received: the structure of inputs=('*', '*')
  warnings.warn(
I0000 00:00:1745946744.815467   62926 service.cc:146] XLA service 0x768f98024b70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745946744.815576   62926 service.cc:154]   StreamExecutor device (0): NVIDIA A800 80GB PCIe, Compute Capability 8.0
2025-04-30 01:12:25.338394: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-30 01:12:26.213892: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90501
Could not load symbol cuFuncGetName. Error: /lib/x86_64-linux-gnu/libcuda.so.1: undefined symbol: cuFuncGetName


 8/85 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - AUC: 0.4739 - Precision: 0.5322 - Recall: 0.5127 - accuracy: 0.4780 - loss: 0.6950

I0000 00:00:1745946757.580214   62926 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


84/85 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - AUC: 0.5015 - Precision: 0.5095 - Recall: 0.5743 - accuracy: 0.5020 - loss: 0.6937

2025-04-30 01:12:40.512996: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5286', 40 bytes spill stores, 40 bytes spill loads



85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - AUC: 0.5015 - Precision: 0.5094 - Recall: 0.5743 - accuracy: 0.5020 - loss: 0.6937

2025-04-30 01:12:56.297003: W tensorflow/core/framework/op_kernel.cc:1840] OP_REQUIRES failed at xla_ops.cc:577 : INTERNAL: ptxas exited with non-zero error code 2, output: : If the error message indicates that a file could not be written, please verify that sufficient filesystem space is provided.
2025-04-30 01:12:56.297085: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INTERNAL: ptxas exited with non-zero error code 2, output: : If the error message indicates that a file could not be written, please verify that sufficient filesystem space is provided.
	 [[{{node StatefulPartitionedCall}}]]


RuntimeError: pybind11::error_already_set: MISMATCH of original and normalized active exception types: ORIGINAL InternalError REPLACED BY KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/framework/errors_impl.py(462): __init__
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/eager/execute.py(53): quick_execute
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/eager/context.py(1552): call_function
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/eager/polymorphic_function/atomic_function.py(251): call_flat
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/eager/polymorphic_function/atomic_function.py(216): call_preflattened
  /data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/tensorflow/python/eager/polymorphic_function/concrete_function.py(1322): _call_flat
  /dat

In [ ]:
hybridmodel.load_weights(checkpoint_filepath)

In [ ]:
# probabilities = hybridmodel.predict([Xeeg_testS_3D, Xfnirs_testS_3D])

In [ ]:
# y_preds= np.round(probabilities)

In [ ]:
# #confusion matrix
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# sns.set()

# plt.figure(figsize=(4,4))

# mat = confusion_matrix(Y_test, y_preds )
# labels = [0,1]

# sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False, cmap='Blues',
#             xticklabels=labels, yticklabels=labels)

# plt.xlabel('Predicted label')
# plt.ylabel('Actual label')

In [ ]:
# #model performance on unseen data
# loss,accuracy,precision,recall,AUC = hybridmodel.evaluate([Xeeg_testS_3D,Xfnirs_testS_3D], Y_test)

In [ ]:
# print("baseline model=", baseline_model)
# print("resnet=", resnet)
# print("seed=",RANDOM_STATE)
# print("lr=", lr)